# Stage 2: SPAM2020 crop-share models

このノートブックは、添付された cropland_soil_group_wetland_excluded_oof_shap_target2020.ipynb のパス、変数変換、空間ブロック分割の考え方を引き継いだ、第2段階（作物選択・作物シェア）の最初の比較用モデルです。

行うこと:

1. SPAM2020 V2r2 の全作物の physical area を読み込む
2. 各グリッドで share_ic = physical_area_ic / sum_k physical_area_ik を作る
3. 同じ説明変数で、線形 fractional-softmax と非線形 LightGBM score model を空間OOFで比較する
4. 係数、予測シェア、評価指標、非線形モデルのSHAP要約を保存する

注意:

- この最初の版の目的変数は「SPAMに面積がある作物の中での条件付きシェア」です。つまり、SPAMの全作物面積を分母にします。
- cropland fraction との残差（未配分地・その他作物）を含む厳密な全土地利用シェアは、次の段階で追加できるように spam_coverage を保存します。
- physical area は作物の作付面積（ha）であり、収量ではありません。
- SPAMの値は観測値そのものではなく、統計値と空間データを組み合わせて推計されたデータです。

In [ ]:
from pathlib import Path
import gc
import json
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.optimize import minimize
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Server paths: copied from the reference cropland notebook where possible
# ---------------------------------------------------------------------
ROOT = Path("/work/tsuda")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
WX_CACHE_DIR = GAEZ_DIR / "CroplandRegression" / "spatial_wx_comparison"
LAND_MASK_DIR = GAEZ_DIR / "LandMasks"
DERIVED_MASK_DIR = LAND_MASK_DIR / "derived_5min"

SPAM_DIR = GAEZ_DIR / "SPAM2020"
OUTPUT_DIR = GAEZ_DIR / "CroplandRegression" / "stage2_spam_crop_share_simple_target2020"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2020
RANDOM_SEED = 42
N_SPLITS = 5
N_CROP_SAMPLE = 30_000
CROPLAND_THRESHOLD = 0.01
SHARE_EPS = 1e-6
REFERENCE_CROP = "RICE"
N_JOBS = 4

# True: exactly the soil-group version of the reference cropland notebook.
# False: use BASE_FEATURES only.
USE_SOIL_GROUP = True

print("OUTPUT_DIR:", OUTPUT_DIR)
print("SPAM_DIR:", SPAM_DIR)

In [ ]:
def load_cache(*names):
    """Load the first existing .npy file from FEATURE_CACHE."""
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError(
        "None of the cache files exists:\n" + "\n".join(str(FEATURE_CACHE / n) for n in names)
    )


def safe_log1p(x):
    x = np.asarray(x, dtype=np.float32)
    return np.log1p(np.maximum(x, 0.0)).astype(np.float32)


def positive_raw(x):
    x = np.asarray(x, dtype=np.float32)
    out = x.copy()
    out[~np.isfinite(out)] = np.nan
    out[out < 0] = np.nan
    return out


def build_or_load_mode_cache(source_path, output_path, target_shape):
    """Aggregate a high-resolution categorical raster to the model grid."""
    if output_path.exists():
        arr = np.load(output_path, mmap_mode="r")
        if tuple(arr.shape) == tuple(target_shape):
            return arr

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(source_path) as src:
        data = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.mode,
            masked=True,
        )
        arr = data.filled(0).astype(np.int16)

    np.save(output_path, arr)
    return np.load(output_path, mmap_mode="r")


def take_grid(arr, rows, cols):
    return np.asarray(arr[rows, cols])


# ---------------------------------------------------------------------
# Grid and the same predictor construction as the cropland notebook
# ---------------------------------------------------------------------
lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
GRID_SHAPE = (len(lat), len(lon))
print("GRID_SHAPE:", GRID_SHAPE)

cropland_cube = np.load(
    CROPLAND_DIR / "cropland_fraction_1950_2024.npy",
    mmap_mode="r",
)
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland = np.asarray(cropland_cube[year_index], dtype=np.float32).copy()
cropland[~np.isfinite(cropland)] = np.nan

# The reference notebook uses these layers for the valid-land mask.
population = load_cache("population_density_2024.npy")
elevation = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")

SOIL_TIF = LAND_MASK_DIR / "gaez_v5_wrb_soil_group_30sec.tif"
EXCLUSION_TIF = LAND_MASK_DIR / "gaez_v5_exclusion_30sec.tif"
soil_group = build_or_load_mode_cache(
    SOIL_TIF,
    DERIVED_MASK_DIR / "soil_group_5min_mode.npy",
    GRID_SHAPE,
)
exclusion = build_or_load_mode_cache(
    EXCLUSION_TIF,
    DERIVED_MASK_DIR / "exclusion_5min_mode.npy",
    GRID_SHAPE,
)

city_time = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas = np.load(
    GLOFAS_DIR / "p10_discharge_max_5min_2020.npy",
    mmap_mode="r",
)
river_distance = np.load(
    GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy",
    mmap_mode="r",
)

rainfed_calorie = load_cache(
    "rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "rainfed_calorie_top5_checked_36crops.npy",
)
irrigated_calorie = load_cache(
    "irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "irrigated_calorie_top5_checked_36crops.npy",
)
wx_50km_rainfed_calorie = load_cache(
    "wx_50km_rainfed_calorie_top5_raw.npy",
)

# Same variable set as the reference cropland notebook.
BASE_FEATURES = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "rainfed_calorie_top5_raw",
    "irrigation_calorie_gain_top5_raw",
    "wx_50km_rainfed_calorie_top5_raw",
    "log_distance_river_gt10_2020",
    "log_glofas_p10_2020",
]
SOIL_FEATURES = BASE_FEATURES + ["soil_group_class"]
MODEL_FEATURES = SOIL_FEATURES if USE_SOIL_GROUP else BASE_FEATURES

CATEGORICAL_FEATURES = [
    f for f in ["exclusion_class", "soil_group_class"]
    if f in MODEL_FEATURES
]
NUMERIC_FEATURES = [
    f for f in MODEL_FEATURES
    if f not in CATEGORICAL_FEATURES
]

print("MODEL_FEATURES:")
print(MODEL_FEATURES)

In [ ]:
def build_feature_frame(rows, cols):
    """Build a DataFrame using the exact transformations used in the reference model."""
    rain = positive_raw(take_grid(rainfed_calorie, rows, cols))
    irr = positive_raw(take_grid(irrigated_calorie, rows, cols))
    wx = positive_raw(take_grid(wx_50km_rainfed_calorie, rows, cols))

    # The reference notebook defines the irrigation gain after replacing
    # invalid values with NaN and clipping negative gains to zero.
    irr_gain = irr - rain
    irr_gain[irr_gain < 0] = 0

    frame = pd.DataFrame({
        "row": rows.astype(np.int32),
        "col": cols.astype(np.int32),
        "lat": lat[rows].astype(np.float32),
        "lon": lon[cols].astype(np.float32),
        "cropland_fraction": take_grid(cropland, rows, cols).astype(np.float32),
        "elevation_m": take_grid(elevation, rows, cols).astype(np.float32),
        "slope": take_grid(slope, rows, cols).astype(np.float32),
        "exclusion_class": take_grid(exclusion, rows, cols).astype(np.int16),
        "soil_group_class": take_grid(soil_group, rows, cols).astype(np.int16),
        "log_city_time_20k_min": safe_log1p(take_grid(city_time, rows, cols)),
        "log_port_time_any_min": safe_log1p(take_grid(port_time, rows, cols)),
        "rainfed_calorie_top5_raw": rain,
        "irrigation_calorie_gain_top5_raw": irr_gain.astype(np.float32),
        "wx_50km_rainfed_calorie_top5_raw": wx,
        "log_distance_river_gt10_2020": safe_log1p(
            take_grid(river_distance, rows, cols)
        ),
        "log_glofas_p10_2020": safe_log1p(take_grid(glofas, rows, cols)),
    })

    # Keep only columns needed by this stage plus location/diagnostics.
    return frame


# Valid land/cropland mask follows the reference notebook.
land_mask = (
    np.isfinite(cropland)
    & (cropland > CROPLAND_THRESHOLD)
    & np.isfinite(np.asarray(population))
    & np.isfinite(np.asarray(elevation))
    & np.isfinite(np.asarray(slope))
)
print("Valid cropland cells:", int(land_mask.sum()))

In [ ]:
# Discover all SPAM2020 crop rasters recursively.
# Preferred target: total physical area (_A_). If _A_ is missing for a crop,
# the notebook sums irrigated (_I_) and rainfed (_R_) maps.
SPAM_PATTERN = re.compile(
    r"_A_(?P<crop>[A-Z0-9]+)_(?P<system>[AIR])\.(?:tif|tiff)$",
    flags=re.IGNORECASE,
)

spam_files = sorted(
    p for p in SPAM_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
)

spam_paths = {}
for path in spam_files:
    match = SPAM_PATTERN.search(path.name)
    if not match:
        continue
    crop = match.group("crop").upper()
    system = match.group("system").upper()
    spam_paths.setdefault(crop, {})[system] = path

if not spam_paths:
    raise FileNotFoundError(
        f"No SPAM crop rasters matching _A_<CROP>_<I/R/A>.tif were found below {SPAM_DIR}"
    )

crop_codes = sorted(spam_paths)
if REFERENCE_CROP not in crop_codes:
    raise ValueError(f"Reference crop {REFERENCE_CROP!r} is not in {crop_codes}")

print("Number of crop codes:", len(crop_codes))
print(crop_codes)

# Check one raster's georeferencing and dimensions.
reference_path = next(iter(spam_paths[REFERENCE_CROP].values()))
with rasterio.open(reference_path) as src:
    spam_shape = (src.height, src.width)
    spam_transform = src.transform
    spam_crs = src.crs
    print("SPAM raster shape:", spam_shape)
    print("SPAM CRS:", spam_crs)
    print("SPAM transform:", spam_transform)

if spam_shape != GRID_SHAPE:
    raise ValueError(
        f"SPAM shape {spam_shape} != model grid shape {GRID_SHAPE}. "
        "Regrid SPAM before running this notebook."
    )


def read_spam_area(crop):
    """Read total physical area in ha for one crop."""
    paths = spam_paths[crop]

    if "A" in paths:
        with rasterio.open(paths["A"]) as src:
            arr = src.read(1, masked=True).filled(0).astype(np.float32)
    else:
        arr = np.zeros(GRID_SHAPE, dtype=np.float32)
        for system in ("I", "R"):
            if system in paths:
                with rasterio.open(paths[system]) as src:
                    part = src.read(1, masked=True).filled(0).astype(np.float32)
                part[~np.isfinite(part)] = 0
                part[part < 0] = 0
                arr += part

    arr[~np.isfinite(arr)] = 0
    arr[arr < 0] = 0
    return arr

print("Raster discovery complete.")

In [ ]:
# Read all total physical-area maps once to construct the target denominator.
# Each individual raster is released after it is added to the total.
total_spam_area = np.zeros(GRID_SHAPE, dtype=np.float32)

for i, crop in enumerate(crop_codes, start=1):
    area = read_spam_area(crop)
    total_spam_area += area
    del area
    if i % 10 == 0 or i == len(crop_codes):
        print(f"{i}/{len(crop_codes)} total-area rasters processed")

total_path = OUTPUT_DIR / "spam2020_total_physical_area_all_crops_ha.npy"
np.save(total_path, total_spam_area)
print("Saved:", total_path)
print("Total SPAM area (ha), positive cells:", int((total_spam_area > 0).sum()))

In [ ]:
# Sample valid cells for the first model run.
# Sampling is spatially reproducible and uses all cells with positive SPAM area
# as the candidate population.
candidate_flat = np.flatnonzero(
    land_mask.ravel()
    & (total_spam_area.ravel() > 0)
)
if len(candidate_flat) == 0:
    raise ValueError("No valid cells have positive SPAM area.")

rng = np.random.default_rng(RANDOM_SEED)
n_sample = min(N_CROP_SAMPLE, len(candidate_flat))
selected_flat = np.sort(
    rng.choice(candidate_flat, size=n_sample, replace=False)
)
rows, cols = np.unravel_index(selected_flat, GRID_SHAPE)

# Read the same grid cells from every crop map.
area_sample = np.zeros((n_sample, len(crop_codes)), dtype=np.float32)
for j, crop in enumerate(crop_codes):
    area = read_spam_area(crop)
    area_sample[:, j] = area[rows, cols]
    del area

total_area_sample = area_sample.sum(axis=1)
valid_target = np.isfinite(total_area_sample) & (total_area_sample > 0)
area_sample = area_sample[valid_target]
rows = rows[valid_target]
cols = cols[valid_target]
total_area_sample = total_area_sample[valid_target]

# Conditional crop shares among the 46 SPAM crop categories.
y_share = area_sample / total_area_sample[:, None]
y_share = np.nan_to_num(y_share, nan=0.0, posinf=0.0, neginf=0.0)
y_share = y_share / y_share.sum(axis=1, keepdims=True)

sample = build_feature_frame(rows, cols)
sample["total_spam_area_ha"] = total_area_sample.astype(np.float32)

# Compare SPAM crop area with the cropland mask as a diagnostic only.
grid_area_path = GAEZ_DIR / "grid_area_2160x4320_ha.npy"
if grid_area_path.exists():
    grid_area_ha = np.load(grid_area_path, mmap_mode="r")
else:
    grid_area_km2_path = CROPLAND_DIR / "grid_area_km2.npy"
    if not grid_area_km2_path.exists():
        raise FileNotFoundError("No grid_area_2160x4320_ha.npy or grid_area_km2.npy found")
    grid_area_ha = np.load(grid_area_km2_path, mmap_mode="r") * 100.0

if tuple(grid_area_ha.shape) != GRID_SHAPE:
    raise ValueError("Grid-area raster has a different shape.")

cropland_area_sample = (
    grid_area_ha[rows, cols].astype(np.float32)
    * sample["cropland_fraction"].to_numpy(dtype=np.float32)
)
sample["spam_coverage"] = (
    sample["total_spam_area_ha"].to_numpy(dtype=np.float32)
    / np.maximum(cropland_area_sample, 1e-6)
).astype(np.float32)

# Drop rows with missing predictors, keeping the target matrix aligned.
valid_features = np.isfinite(
    sample[NUMERIC_FEATURES].to_numpy(dtype=np.float32)
).all(axis=1)
valid_features &= sample[CATEGORICAL_FEATURES].notna().all(axis=1).to_numpy()

sample = sample.loc[valid_features].reset_index(drop=True)
y_share = y_share[valid_features]
area_sample = area_sample[valid_features]

# Spatial block definition is the same 10-degree block idea as the reference.
sample["spatial_block"] = (
    np.floor((sample["lat"] + 90.0) / 10.0).astype(np.int32) * 36
    + np.floor((sample["lon"] + 180.0) / 10.0).astype(np.int32)
)

# Use total SPAM physical area as a first-pass weight.
sample_weight = sample["total_spam_area_ha"].to_numpy(dtype=np.float64)
sample_weight = sample_weight / np.maximum(np.nanmean(sample_weight), 1e-12)

print("Final sample:", sample.shape)
print("Target matrix:", y_share.shape)
print("Share-sum range:", y_share.sum(axis=1).min(), y_share.sum(axis=1).max())
print("SPAM coverage quantiles:")
print(sample["spam_coverage"].quantile([0, .01, .05, .5, .95, .99, 1.0]))

In [ ]:
# Save a compact target/diagnostic table and the full share matrix.
diagnostic_cols = [
    "row", "col", "lat", "lon", "cropland_fraction",
    "total_spam_area_ha", "spam_coverage", "spatial_block",
]
dominant_index = np.argmax(y_share, axis=1)
sample["dominant_crop"] = np.asarray(crop_codes, dtype=object)[dominant_index]
sample["dominant_share"] = y_share[np.arange(len(y_share)), dominant_index].astype(np.float32)

sample[diagnostic_cols + ["dominant_crop", "dominant_share"]].to_csv(
    OUTPUT_DIR / "stage2_spam_sample_diagnostics.csv.gz",
    index=False,
    compression="gzip",
)

share_columns = [f"share_{crop}" for crop in crop_codes]
share_table = pd.DataFrame(y_share, columns=share_columns)
share_table = pd.concat(
    [sample[diagnostic_cols].reset_index(drop=True), share_table],
    axis=1,
)
share_table.to_csv(
    OUTPUT_DIR / "stage2_spam_sample_conditional_shares.csv.gz",
    index=False,
    compression="gzip",
)

dominant_counts = sample["dominant_crop"].value_counts().rename("n_cells")
display(dominant_counts.head(20).to_frame())
print("Saved target tables to:", OUTPUT_DIR)

## モデル定義

線形モデルは、作物ごとの効用を

\[
U_{ic} = \alpha_c + X_i^\top\beta_c
\]

と置き、

\[
\hat{s}_{ic}
= \frac{\exp(U_{ic})}{\sum_k \exp(U_{ik})}
\]

を、SPAMの fractional share に対する cross-entropy で直接推定します。識別のため、RICE の効用を基準（係数ゼロ）にします。したがって係数は「RICEに対する相対効用」の係数です。

非線形モデルは、同じグリッドを46作物分に展開し、crop_code とグリッド説明変数を使って

\[
g(i,c) \simeq \log(s_{ic}+\epsilon)
\]

をLightGBMで推定し、得られたスコアをsoftmaxで正規化します。これは最初の比較用の非線形モデルです。

In [ ]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        # Compatibility with older scikit-learn versions.
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_linear_design(train_frame, test_frame):
    scaler = StandardScaler()
    x_num_train = scaler.fit_transform(
        train_frame[NUMERIC_FEATURES].to_numpy(dtype=np.float64)
    )
    x_num_test = scaler.transform(
        test_frame[NUMERIC_FEATURES].to_numpy(dtype=np.float64)
    )

    if CATEGORICAL_FEATURES:
        encoder = make_encoder()
        x_cat_train = encoder.fit_transform(
            train_frame[CATEGORICAL_FEATURES].astype(str)
        )
        x_cat_test = encoder.transform(
            test_frame[CATEGORICAL_FEATURES].astype(str)
        )
        cat_names = list(encoder.get_feature_names_out(CATEGORICAL_FEATURES))
    else:
        encoder = None
        x_cat_train = np.empty((len(train_frame), 0), dtype=np.float64)
        x_cat_test = np.empty((len(test_frame), 0), dtype=np.float64)
        cat_names = []

    x_train = np.column_stack([x_num_train, x_cat_train]).astype(np.float64)
    x_test = np.column_stack([x_num_test, x_cat_test]).astype(np.float64)
    feature_names = NUMERIC_FEATURES + cat_names
    return x_train, x_test, feature_names, scaler, encoder


def stable_softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(np.clip(shifted, -50, 50))
    return exp_scores / np.maximum(exp_scores.sum(axis=1, keepdims=True), 1e-300)


def fit_linear_softmax(x, y, weights, crop_codes, reference_crop=REFERENCE_CROP,
                       l2=0.02, maxiter=200):
    """Fit fractional multinomial softmax with one reference crop."""
    n, p = x.shape
    k = len(crop_codes)
    ref_idx = crop_codes.index(reference_crop)
    non_ref = np.asarray([j for j in range(k) if j != ref_idx], dtype=np.int32)
    x_aug = np.column_stack([np.ones(n), x])
    w = np.asarray(weights, dtype=np.float64)
    w = w / np.maximum(w.mean(), 1e-12)
    y = np.asarray(y, dtype=np.float64)

    def objective_and_gradient(theta):
        beta = theta.reshape(k - 1, p + 1)
        scores = np.zeros((n, k), dtype=np.float64)
        scores[:, non_ref] = x_aug @ beta.T
        probs = stable_softmax(scores)

        weighted_y = w[:, None] * y
        loss = -np.sum(weighted_y * np.log(np.clip(probs, 1e-12, 1.0))) / np.sum(w)
        loss += 0.5 * l2 * np.sum(beta[:, 1:] ** 2)

        score_gradient = (w[:, None] * (probs - y)) / np.sum(w)
        beta_gradient = score_gradient[:, non_ref].T @ x_aug
        beta_gradient[:, 1:] += l2 * beta[:, 1:]
        return float(loss), beta_gradient.ravel()

    initial = np.zeros((k - 1, p + 1), dtype=np.float64)
    result = minimize(
        fun=lambda theta: objective_and_gradient(theta)[0],
        x0=initial.ravel(),
        jac=lambda theta: objective_and_gradient(theta)[1],
        method="L-BFGS-B",
        options={"maxiter": maxiter, "ftol": 1e-8},
    )

    return {
        "beta": result.x.reshape(k - 1, p + 1),
        "non_ref": non_ref,
        "reference_crop": reference_crop,
        "crop_codes": list(crop_codes),
        "feature_names": ["intercept"] + list(
            [f"num__{f}" for f in NUMERIC_FEATURES]
            + [f"cat__{f}" for f in []]
        ),
        "optimizer_result": result,
    }


def predict_linear_softmax(model, x):
    beta = model["beta"]
    non_ref = model["non_ref"]
    x_aug = np.column_stack([np.ones(len(x)), x])
    scores = np.zeros((len(x), len(model["crop_codes"])), dtype=np.float64)
    scores[:, non_ref] = x_aug @ beta.T
    return stable_softmax(scores)

In [ ]:
def make_long_frame(frame, y=None, weights=None, crop_codes=crop_codes):
    """Expand grid rows into (grid, crop) rows for pooled nonlinear fitting."""
    n = len(frame)
    k = len(crop_codes)
    repeated_index = np.repeat(np.arange(n), k)

    long_frame = frame.iloc[repeated_index][MODEL_FEATURES].reset_index(drop=True).copy()
    long_frame["crop_code"] = np.tile(np.asarray(crop_codes, dtype=object), n)
    long_frame["crop_code"] = pd.Categorical(
        long_frame["crop_code"], categories=crop_codes
    )

    for col in CATEGORICAL_FEATURES:
        if col == "exclusion_class":
            levels = list(range(0, 20))
        else:
            levels = list(range(0, 100))
        long_frame[col] = pd.Categorical(long_frame[col], categories=levels)

    if y is not None:
        long_frame["target_log_share"] = np.log(
            np.asarray(y, dtype=np.float64).reshape(-1) + SHARE_EPS
        )
    if weights is not None:
        long_frame["sample_weight"] = np.repeat(
            np.asarray(weights, dtype=np.float64), k
        )
    return long_frame


def fit_nonlinear_score_model(train_frame, y_train, weights):
    long_train = make_long_frame(train_frame, y_train, weights)
    categorical_for_lgbm = ["crop_code"] + CATEGORICAL_FEATURES

    model = LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbosity=-1,
    )
    model.fit(
        long_train[MODEL_FEATURES + ["crop_code"]],
        long_train["target_log_share"],
        sample_weight=long_train["sample_weight"],
        categorical_feature=categorical_for_lgbm,
    )
    del long_train
    gc.collect()
    return model


def predict_nonlinear_score_model(model, frame, crop_codes=crop_codes):
    long_test = make_long_frame(frame, y=None, crop_codes=crop_codes)
    score = model.predict(long_test[MODEL_FEATURES + ["crop_code"]])
    score = np.asarray(score, dtype=np.float64).reshape(len(frame), len(crop_codes))
    del long_test
    return stable_softmax(score)

In [ ]:
def weighted_cross_entropy(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    return float(
        -np.sum(w[:, None] * y_true * np.log(np.clip(y_pred, 1e-12, 1.0)))
        / np.sum(w)
    )


def weighted_share_rmse(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    mse_by_row = np.mean((y_true - y_pred) ** 2, axis=1)
    return float(np.sqrt(np.sum(w * mse_by_row) / np.sum(w)))


def dominant_accuracy(y_true, y_pred, weights=None):
    true_idx = np.argmax(y_true, axis=1)
    pred_idx = np.argmax(y_pred, axis=1)
    hit = (true_idx == pred_idx).astype(np.float64)
    if weights is None:
        return float(hit.mean())
    return float(np.average(hit, weights=weights))


def evaluate_predictions(name, y_true, y_pred, weights):
    return {
        "model": name,
        "weighted_cross_entropy": weighted_cross_entropy(y_true, y_pred, weights),
        "weighted_share_rmse": weighted_share_rmse(y_true, y_pred, weights),
        "dominant_crop_accuracy": dominant_accuracy(y_true, y_pred, weights),
    }

In [ ]:
# Spatial out-of-fold comparison.
# Each fold refits preprocessing, the linear model, and the nonlinear model.
groups = sample["spatial_block"].to_numpy()
splitter = GroupKFold(n_splits=N_SPLITS)

linear_oof = np.full_like(y_share, np.nan, dtype=np.float32)
nonlinear_oof = np.full_like(y_share, np.nan, dtype=np.float32)
fold_metrics = []

for fold_number, (train_idx, test_idx) in enumerate(
    splitter.split(sample, groups=groups), start=1
):
    print(f"--- fold {fold_number}/{N_SPLITS} ---")
    train_frame = sample.iloc[train_idx].copy()
    test_frame = sample.iloc[test_idx].copy()
    y_train = y_share[train_idx]
    y_test = y_share[test_idx]
    w_train = sample_weight[train_idx]
    w_test = sample_weight[test_idx]

    # Linear fractional-softmax regression.
    x_train, x_test, design_names, scaler, encoder = make_linear_design(
        train_frame, test_frame
    )
    linear_model = fit_linear_softmax(
        x_train,
        y_train,
        w_train,
        crop_codes,
        reference_crop=REFERENCE_CROP,
        l2=0.02,
        maxiter=200,
    )
    pred_linear = predict_linear_softmax(linear_model, x_test)
    linear_oof[test_idx] = pred_linear.astype(np.float32)

    # Nonlinear pooled LightGBM score model followed by softmax normalization.
    nonlinear_model = fit_nonlinear_score_model(train_frame, y_train, w_train)
    pred_nonlinear = predict_nonlinear_score_model(nonlinear_model, test_frame)
    nonlinear_oof[test_idx] = pred_nonlinear.astype(np.float32)

    fold_metrics.append({
        "fold": fold_number,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        **{
            f"linear_{key}": value
            for key, value in evaluate_predictions(
                "linear", y_test, pred_linear, w_test
            ).items()
            if key != "model"
        },
        **{
            f"nonlinear_{key}": value
            for key, value in evaluate_predictions(
                "nonlinear", y_test, pred_nonlinear, w_test
            ).items()
            if key != "model"
        },
    })

    del train_frame, test_frame, x_train, x_test
    del linear_model, nonlinear_model, pred_linear, pred_nonlinear
    gc.collect()

metrics_df = pd.DataFrame(fold_metrics)
display(metrics_df)
display(metrics_df.mean(numeric_only=True).to_frame("mean").T)

metrics_df.to_csv(
    OUTPUT_DIR / "stage2_spam_crop_share_spatial_oof_metrics.csv",
    index=False,
)
np.save(OUTPUT_DIR / "stage2_spam_crop_share_linear_oof.npy", linear_oof)
np.save(OUTPUT_DIR / "stage2_spam_crop_share_nonlinear_oof.npy", nonlinear_oof)
print("OOF predictions and metrics saved.")

In [ ]:
# Overall OOF metrics and a simple global-share baseline.
global_share = np.average(y_share, axis=0, weights=sample_weight)
global_pred = np.broadcast_to(global_share[None, :], y_share.shape)

overall_metrics = pd.DataFrame([
    evaluate_predictions("global_share_baseline", y_share, global_pred, sample_weight),
    evaluate_predictions("linear_fractional_softmax", y_share, linear_oof, sample_weight),
    evaluate_predictions("nonlinear_lightgbm_score_softmax", y_share, nonlinear_oof, sample_weight),
])
display(overall_metrics)

overall_metrics.to_csv(
    OUTPUT_DIR / "stage2_spam_crop_share_overall_oof_metrics.csv",
    index=False,
)

In [ ]:
# Fit the linear model on all sampled cells and save relative utility coefficients.
x_all, _, design_names, scaler_all, encoder_all = make_linear_design(sample, sample)
linear_full = fit_linear_softmax(
    x_all,
    y_share,
    sample_weight,
    crop_codes,
    reference_crop=REFERENCE_CROP,
    l2=0.02,
    maxiter=250,
)

# Reconstruct names exactly, including one-hot categories from the fitted encoder.
if encoder_all is not None:
    full_design_names = NUMERIC_FEATURES + list(
        encoder_all.get_feature_names_out(CATEGORICAL_FEATURES)
    )
else:
    full_design_names = list(NUMERIC_FEATURES)

coef_names = ["intercept"] + full_design_names
coef_table = pd.DataFrame(
    linear_full["beta"],
    index=[c for c in crop_codes if c != REFERENCE_CROP],
    columns=coef_names,
)
coef_table.index.name = "crop_relative_to_reference"
coef_table.to_csv(
    OUTPUT_DIR / "stage2_linear_softmax_relative_utility_coefficients.csv"
)

print("Reference crop:", REFERENCE_CROP)
display(coef_table.head())
print("Saved linear coefficients.")

In [ ]:
# Optional SHAP summary for the final nonlinear pooled model.
# SHAP is computed for the model's log-share score, before softmax normalization.
try:
    import shap

    nonlinear_full = fit_nonlinear_score_model(sample, y_share, sample_weight)
    long_for_shap = make_long_frame(sample, y=None)
    shap_n = min(20_000, len(long_for_shap))
    shap_rng = np.random.default_rng(RANDOM_SEED + 1000)
    shap_idx = np.sort(
        shap_rng.choice(len(long_for_shap), size=shap_n, replace=False)
    )
    shap_frame = long_for_shap.iloc[shap_idx][MODEL_FEATURES + ["crop_code"]]

    explainer = shap.TreeExplainer(nonlinear_full)
    shap_values = explainer.shap_values(shap_frame)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.asarray(shap_values)

    shap_feature_names = MODEL_FEATURES + ["crop_code"]
    shap_summary = pd.DataFrame({
        "feature": shap_feature_names,
        "mean_abs_shap_log_share": np.nanmean(np.abs(shap_values), axis=0),
        "mean_signed_shap_log_share": np.nanmean(shap_values, axis=0),
    }).sort_values("mean_abs_shap_log_share", ascending=False)

    shap_summary.to_csv(
        OUTPUT_DIR / "stage2_nonlinear_shap_feature_summary.csv",
        index=False,
    )

    feature_group = {
        "elevation_m": "land_soil",
        "slope": "land_soil",
        "soil_group_class": "land_soil",
        "exclusion_class": "land_soil",
        "log_city_time_20k_min": "market_access",
        "log_port_time_any_min": "market_access",
        "rainfed_calorie_top5_raw": "climate_water",
        "irrigation_calorie_gain_top5_raw": "climate_water",
        "wx_50km_rainfed_calorie_top5_raw": "climate_water",
        "log_distance_river_gt10_2020": "climate_water",
        "log_glofas_p10_2020": "climate_water",
        "crop_code": "crop_identity_control",
    }
    shap_summary["group"] = shap_summary["feature"].map(feature_group)
    group_summary = (
        shap_summary[shap_summary["group"] != "crop_identity_control"]
        .groupby("group", as_index=False)["mean_abs_shap_log_share"]
        .sum()
        .sort_values("mean_abs_shap_log_share", ascending=False)
    )
    group_summary.to_csv(
        OUTPUT_DIR / "stage2_nonlinear_shap_group_summary.csv",
        index=False,
    )

    display(shap_summary)
    display(group_summary)
except ImportError:
    print("shap is not installed; skipped SHAP summary.")

In [ ]:
metadata = {
    "year": YEAR,
    "spam_dir": str(SPAM_DIR),
    "n_crops": len(crop_codes),
    "crop_codes": crop_codes,
    "spam_area_unit": "ha",
    "target": "conditional_share_among_SPAM_crops",
    "share_formula": "physical_area_ic / sum_k(physical_area_ik)",
    "reference_crop_for_linear_model": REFERENCE_CROP,
    "model_features": MODEL_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "n_sample": int(len(sample)),
    "n_splits": N_SPLITS,
    "spatial_block": "10-degree latitude/longitude blocks",
    "sample_weight": "total_spam_area_ha divided by sample mean",
    "linear_model": "fractional cross-entropy softmax with crop-specific coefficients",
    "nonlinear_model": "pooled LightGBM regression of log share followed by softmax",
}
with open(OUTPUT_DIR / "stage2_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("All outputs are in:", OUTPUT_DIR)
print("Next modeling step: add a residual/Other category or use a two-part target so that")
print("the denominator is cropland area rather than only the SPAM-positive crop total.")